In [2]:
import numpy as np
import jax
import jax.numpy as jnp
import flax.serialization
import matplotlib.pyplot as plt

# 1) Load your data
train_data = np.load("data/spm/for_soc/test/GRF_2200.npz")
test_data = np.load("data/spm/for_soc/test/GRF_2200.npz")

train_I = jnp.array(train_data["current"])
train_c0_anode = jnp.array(train_data["initial_concentration_anode"])
train_cn_anode = jnp.array(train_data["target_concentration_anode"])
# ... likewise for cathode & test sets

# 2) Define or import your FNO model class
from FNO import FNO

# 3) Build anode & cathode models, load saved params
model_anode = FNO(k_modes=10, fno_depth=6, hidden_channels=64, output_channels=1)
model_cathode = FNO(k_modes=10, fno_depth=6, hidden_channels=64, output_channels=1)

def load_fno_params(model, file_path, key):
    """
    Initialize your model randomly, then load the trained bytes from file_path.
    """
    # build a dummy input
    dummy_input = jnp.zeros((1, 24, 85, 4))
    init_params = model.init(key, dummy_input)
    with open(file_path, "rb") as f:
        raw_bytes = f.read()
    params_loaded = flax.serialization.from_bytes(init_params, raw_bytes)
    return params_loaded

rng = jax.random.PRNGKey(42)
anode_file = "trained_models/diff_D/anode_GRF__2025-03-27_12-15-16.msgpack"
cathode_file = "trained_models/diff_D/cathode_GRF__2025-03-27_13-13-25.msgpack"
params_anode = load_fno_params(model_anode, anode_file, rng)
params_cathode = load_fno_params(model_cathode, cathode_file, rng)

# 4) Preprocess function: turn (I, c0, cn) -> padded (X, Y)
def preprocess_data(I_array, c0_array, cn_array, 
                    t_lin, r_lin, padding_t, padding_r):
    # code similar to your snippet, but pass t_lin, r_lin as arguments
    # returns X, Y
    ...
    return X, Y

# 5) Actually call preprocess_data
padding_t, padding_r = 5, 2
t_max = 900
num_samples_I, num_samples_c0 = 75, 20
t = np.linspace(0, t_max, num_samples_I)
r = np.linspace(0, 1, num_samples_c0)
t_lin = jnp.array(t / t_max)
r_lin = jnp.array(r)

X_test_anode, Y_test_anode = preprocess_data(test_I, test_c0_anode, test_cn_anode, t_lin, r_lin, padding_t, padding_r)
X_test_cathode, Y_test_cathode = preprocess_data(test_I, test_c0_cathode, test_cn_cathode, t_lin, r_lin, padding_t, padding_r)

# 6) Forward pass on a single sample
test_idx = jax.random.randint(rng, shape=(), minval=0, maxval=X_test_anode.shape[0])
X_a = X_test_anode[test_idx][None, ...]  # shape (1,24,85,4)
X_c = X_test_cathode[test_idx][None, ...]

c_pred_a = model_anode.apply(params_anode, X_a)  # shape (1,24,85,1)
c_pred_c = model_cathode.apply(params_cathode, X_c)

# 7) Un-pad
c_pred_a_unpadded = c_pred_a[0, padding_r:-padding_r, padding_t:-padding_t, 0]  # shape (20,75)
c_pred_c_unpadded = c_pred_c[0, padding_r:-padding_r, padding_t:-padding_t, 0]

# 8) Compare to ground truth
c_true_a = test_cn_anode[test_idx]  # (20,75)
c_true_c = test_cn_cathode[test_idx]  # (20,75)

mae_anode = jnp.mean(jnp.abs(c_pred_a_unpadded - c_true_a))
print("Anode MAE:", mae_anode)

# 9) If desired, compute voltage with a post-processing function
# V_pred, V_true = ...
# diff = jnp.abs(V_pred - V_true)
# etc.


NameError: name 'test_I' is not defined